# 04. Level up: API Prozorro
**Мета:** взяти точну дату договору з API, бо `published_date` іноді оновлюється пізніше.

**Вхід:** `tender_id` з `clean.parquet`; публічне API `public-api.prozorro.gov.ua/api/2.5/tenders/{id}`.

**Що робимо:** завантаження 9 305 записів (8 потоків, 3 спроби, тест на 20) у `data/raw/api_tenders.jsonl`; розбір назв, дат договорів і умов поставки; перевірка проксі: дата реєстрації договору проти `dateSigned` на 1 572 договорах.

**Результат:** 0 помилок; реєстрація відстає від підписання в медіані на 0.4 дня (90% до 2.4 дня).

**Вихід:** `data/processed/api_contracts.parquet`, `data/processed/final.parquet`.

In [1]:
import json, os, time
import requests
import duckdb
from concurrent.futures import ThreadPoolExecutor, as_completed

In [10]:
con = duckdb.connect()
ids = con.sql("SELECT tender_id FROM '../data/processed/clean.parquet'").df()['tender_id'].tolist()

API     = 'https://public-api.prozorro.gov.ua/api/2.5/tenders/'
RAW_API = '../data/raw/api_tenders.jsonl'
KEEP    = ['title', 'status', 'date', 'dateModified', 'procurementMethodType',
           'tenderPeriod', 'items', 'awards', 'contracts', 'milestones']

done = set()
if os.path.exists(RAW_API):
    with open(RAW_API) as f:
        done = {json.loads(line)['tender_id'] for line in f}

todo = [t for t in ids if t not in done]
print(f'всього {len(ids)}, вже є {len(done)}, залишилось {len(todo)}')

всього 9305, вже є 9305, залишилось 0


In [5]:
session = requests.Session()

def fetch(tid):
    for attempt in range(3):
        try:
            r = session.get(API + tid, timeout=30)
            if r.status_code == 200:
                data = r.json()['data']
                return tid, {k: data.get(k) for k in KEEP}
            if r.status_code == 404:
                return tid, None
        except requests.RequestException:
            pass
        time.sleep(2 * (attempt + 1))   # пауза перед повтором: 2, 4, 6 секунд
    return tid, None

def run(batch, workers=8):
    failed = []
    with open(RAW_API, 'a', encoding='utf-8') as out, ThreadPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(fetch, t) for t in batch]
        for i, fut in enumerate(as_completed(futures), 1):
            tid, data = fut.result()
            if data is None:
                failed.append(tid)
            else:
                out.write(json.dumps({'tender_id': tid, 'data': data}, ensure_ascii=False) + '\n')
            if i % 500 == 0:
                print(f'{i} з {len(batch)}')
    print(f'готово: {len(batch) - len(failed)}, помилок: {len(failed)}')
    return failed

In [6]:
failed = run(todo[:20])

готово: 20, помилок: 0


In [9]:
failed = run(todo)

500 з 9285
1000 з 9285
1500 з 9285
2000 з 9285
2500 з 9285
3000 з 9285
3500 з 9285
4000 з 9285
4500 з 9285
5000 з 9285
5500 з 9285
6000 з 9285
6500 з 9285
7000 з 9285
7500 з 9285
8000 з 9285
8500 з 9285
9000 з 9285
готово: 9285, помилок: 0


In [14]:
import pandas as pd

rows = []
with open(RAW_API, encoding='utf-8') as f:
    for line in f:
        rec = json.loads(line)
        d = rec['data']

        # дата підписання: найраніша серед чинних або виконаних договорів
        signed = [pd.to_datetime(c['dateSigned'], utc=True)
                  for c in (d.get('contracts') or [])
                  if c.get('status') in ('active', 'terminated') and c.get('dateSigned')]

        # дата реєстрації чинного договору (проксі підписання, перевірено на 1 572 договорах)
        active = [pd.to_datetime(c['date'], utc=True)
                  for c in (d.get('contracts') or [])
                  if c.get('status') == 'active' and c.get('date')]

        # плановий строк поставки з умов договору (milestones)
        delivery = [m for m in (d.get('milestones') or []) if m.get('type') == 'delivery']

        rows.append({
            'tender_id':      rec['tender_id'],
            'method_type':    d.get('procurementMethodType'),
            'title':          d.get('title'),
            'item_desc':      ((d.get('items') or [{}])[0]).get('description'),
            'date_signed':    min(signed) if signed else None,
            'contract_date':  min(active) if active else None,
            'delivery_days':  (delivery[0].get('duration') or {}).get('days') if delivery else None,
            'delivery_after': delivery[0].get('title') if delivery else None,
        })

api = pd.DataFrame(rows)
api.to_parquet('../data/processed/api_contracts.parquet', index=False)

print(len(api), 'рядків')
print('є дата підписання:', api['date_signed'].notna().sum())
print('є дата договору (проксі):', api['contract_date'].notna().sum())
print('є умови поставки:',  api['delivery_days'].notna().sum())
api.head()

9305 рядків
є дата підписання: 1478
є дата договору (проксі): 9305
є умови поставки: 575


,tender_id,method_type,title,item_desc,date_signed,contract_date,delivery_days,delivery_after
0,2b5eb00e63a245089100b3a964df59cb,belowThreshold,Радари,Передавач візування УК-26М1,2022-03-02 22:01:00+00:00,2022-03-04 07:43:27.178793+00:00,NaN,NaN
1,fb123657f4a148e69ae9f5e347075826,belowThreshold,Радари,Блок перетворення координат УК-69М2,2022-03-02 22:01:00+00:00,2022-03-04 07:35:43.177783+00:00,NaN,NaN
2,10779f70b6e2472182ebb25791c1c1a1,belowThreshold,Радари,Блок підсилювачів приборних приводів УК-66М2,2022-03-02 22:01:00+00:00,2022-03-04 07:56:10.757786+00:00,NaN,NaN
3,042c0b1544934868972b56744f272052,aboveThreshold,Мультикоптер DJI Matrice 30T (в комплект входи...,Мультикоптер DJI Matrice 30T ((в комплект вход...,2022-12-13 08:48:00+00:00,2022-12-14 08:51:15.518351+00:00,NaN,NaN
4,2c22c295a65b4927bf16113fbe738180,aboveThreshold,Навчальна роботизована повітряна платформа (об...,Навчальна роботизована повітряна платформа (об...,2022-11-29 13:00:00+00:00,2022-12-01 08:28:16.100670+00:00,NaN,NaN


In [12]:
from collections import Counter

keys = Counter()
statuses = Counter()
for line in open(RAW_API, encoding='utf-8'):
    d = json.loads(line)['data']
    for c in (d.get('contracts') or []):
        statuses[c.get('status')] += 1
        keys.update(c.keys())

print('статуси договорів:', statuses)
print('поля в договорах:', keys.most_common())

статуси договорів: Counter({'active': 9657, 'cancelled': 1181})
поля в договорах: [('id', 10838), ('contractID', 10838), ('status', 10838), ('date', 10838), ('value', 10838), ('awardID', 10838), ('suppliers', 1669), ('items', 1669), ('contractNumber', 1572), ('period', 1572), ('dateSigned', 1572), ('documents', 1572), ('title', 89), ('description', 40)]


In [13]:
diffs = []
for line in open(RAW_API, encoding='utf-8'):
    d = json.loads(line)['data']
    for c in (d.get('contracts') or []):
        if c.get('status') == 'active' and c.get('dateSigned'):
            diff = pd.to_datetime(c['date'], utc=True) - pd.to_datetime(c['dateSigned'], utc=True)
            diffs.append(diff.total_seconds() / 86400)   # різниця в днях

diffs = pd.Series(diffs)
print('договорів з обома датами:', len(diffs))
print(diffs.describe(percentiles=[0.5, 0.8, 0.9, 0.95]))

договорів з обома датами: 1572
count    1572.000000
mean        0.776195
std         1.127548
min         0.000248
50%         0.424001
80%         1.366101
90%         2.398543
95%         3.372639
max         7.002463
dtype: float64


In [15]:
con.sql("""
    COPY (
        SELECT c.*,
               a.method_type, a.title, a.item_desc,
               a.date_signed, a.contract_date,
               a.delivery_days, a.delivery_after,
               date_diff('day', c.tender_start_date, c.award_date)    AS d_to_award,
               date_diff('day', c.award_date,        a.contract_date) AS d_award_to_contract,
               date_diff('day', c.tender_start_date, a.contract_date) AS d_total
        FROM '../data/processed/clean.parquet' AS c
        JOIN '../data/processed/api_contracts.parquet' AS a USING (tender_id)
    ) TO '../data/processed/final.parquet' (FORMAT parquet)
""")

con.sql("""
    SELECT COUNT(*) AS n,
           SUM(CASE WHEN d_to_award < 0 THEN 1 ELSE 0 END)          AS neg_award,
           SUM(CASE WHEN d_award_to_contract < 0 THEN 1 ELSE 0 END) AS neg_contract,
           SUM(CASE WHEN d_total < 0 THEN 1 ELSE 0 END)             AS neg_total
    FROM '../data/processed/final.parquet'
""").show()

┌───────┬───────────┬──────────────┬───────────┐
│   n   │ neg_award │ neg_contract │ neg_total │
│ int64 │  int128   │    int128    │  int128   │
├───────┼───────────┼──────────────┼───────────┤
│  9305 │         0 │            7 │         0 │
└───────┴───────────┴──────────────┴───────────┘

